# Language Model for Predicting Mutations

## Imports and Setup

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import yaml
import tqdm
import warnings

warnings.filterwarnings("ignore")

## Network Architecture

<img src="./images/architecture.png" width="500">

In [2]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a linear layer followed by a non-linearity """

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head, block_size, dropout):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedFoward(n_embd=n_embd, dropout=dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout, device):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd=n_embd, n_head=n_head, block_size=block_size, dropout=dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.device = device
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=self.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -self.block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

## Dataset

A sequence of mutations in a DFS path in a MAT.

To extract all mutation paths from an input MAT, the following matUtils command is used:

```
matUtils extract --all-paths paths.txt -i input.pb
```

In [3]:
# Each mutation is represented as (position, substituted nucleotide)

mutations_data = [
    (266, 'T'),
    (288, 'T'),
    (303, 'A'),
    (14408, 'C'),
    (14408, 'C'),
    (23403, 'A'),
    (28881, 'A'),
    (28882, 'C'),
    (28883, 'G'),
    (28884, 'C'),
    (28885, 'A'),
    (28886, 'A'),
    (28887, 'C'),
]

## Mapping data to numbers

Each unique element in the dataset is mapped to an integer for input into the model.

In [4]:
from mutations_data import *

TOTAL_BASE_PAIRS = 29903 # number of base pairs in the SARS-CoV-2 genome
NUMBER_OF_BASES = 4 # A, C, G, T

vocab_size = NUMBER_OF_BASES * TOTAL_BASE_PAIRS
base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

encode = lambda mutations: [NUMBER_OF_BASES*mut[0] + base_map[mut[1]] for mut in mutations]
decode = lambda indices: [((index // NUMBER_OF_BASES),list(base_map.keys())[index % NUMBER_OF_BASES]) for index in indices]

In [6]:
encode([(120,'T'),(401,'G')])

[483, 1606]

## Hyperparameters

See **hyperparameters.yaml** for the values of the hyperparameters used.

In [7]:
# hyperparameters
hyperparams = yaml.load(open('hyperparameters.yaml'), Loader=yaml.FullLoader)
batch_size = hyperparams['batch_size']
block_size = hyperparams['block_size']
max_iters = hyperparams['max_iters']
eval_interval = hyperparams['eval_interval']
learning_rate = hyperparams['learning_rate']
eval_iters = hyperparams['eval_iters']
n_embd = hyperparams['n_embd']
n_head = hyperparams['n_head']
n_layer = hyperparams['n_layer']
dropout = hyperparams['dropout']

## Training and Validation

In [8]:
# Set random seed
torch.manual_seed(1337)

In [9]:
# Device - MPS
device = torch.device("mps" if torch.backends.mps.is_available() and torch.backends.mps.is_built() else "cpu")

# Device - CUDA
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# Train and test splits
data = torch.tensor(encode(mutations_data), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [11]:
model = BigramLanguageModel(vocab_size=vocab_size, n_embd=n_embd, n_head=n_head, n_layer=n_layer, block_size=block_size, dropout=dropout, device=device)
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

15.629308 M parameters


In [12]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [13]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in tqdm.tqdm(range(max_iters)):
   
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        tqdm.tqdm.write(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

  0%|          | 0/5000 [00:07<?, ?it/s]

step 0: train loss 11.9096, val loss 11.5975


  2%|▏         | 101/5000 [00:22<1:24:30,  1.03s/it]

step 100: train loss 0.3882, val loss 12.3723


  4%|▍         | 202/5000 [00:38<1:37:05,  1.21s/it]

...

step 4800: train loss 0.1344, val loss 15.3681


 98%|█████████▊| 4901/5000 [12:35<01:47,  1.09s/it]

step 4900: train loss 0.1447, val loss 15.3442


100%|██████████| 5000/5000 [12:49<00:00,  6.49it/s]

step 4999: train loss 0.1459, val loss 15.4854


In [15]:
# Save model
torch.save(model.state_dict(), 'model.pt')
print("Model saved")

Model saved


## Generating from context

In [24]:
context = torch.tensor(encode([(266, 'T'),(288, 'T'),(303, 'A')]), dtype=torch.long, device=device).unsqueeze(0)
print(decode(m.generate(context, max_new_tokens=1)[0].tolist()))

[(266, 'T'), (288, 'T'), (303, 'A'), (14408, 'C')]


## References

* Vaswani, Ashish, et al. "Attention Is All You Need." ArXiv, 2017,  /abs/1706.03762. Accessed 7 Aug. 2023. ([link](https://arxiv.org/pdf/1706.03762.pdf))